In [1]:
import re
import time
import random
import requests
import pandas as pd

from tqdm.notebook import tqdm
from collections import Counter


In [2]:
def wait_random():
    extra = random.uniform(1.5, 2.5)
    total = 1 + extra               
    time.sleep(total)

def clean_address(address: str) -> str:
    """
    Если строка начинается с четырёх и более подряд идущих цифр,
    удаляет их и возвращает оставшуюся часть без ведущих пробелов.
    """
    # ^\d{4,} – начало строки, 4 и более цифр
    # (.*) – захватываем всё, что идёт после этих цифр
    cleaned = re.sub(r'^\d{4,}\s*(.*)', r'\1', address)
    return cleaned

In [3]:
data = pd.read_excel("для встречи (1).xlsx")

In [4]:
data["address"] = data["clean_address"]

In [5]:
data["clean_address"] = data["clean_address"].apply(lambda x: str(x).replace("РОССИЯ,", "").strip())

In [6]:
data["clean_address"] = data["clean_address"].apply(lambda x: str(x).replace(", Д", ",").replace(",Д", ",").strip())

In [7]:
data["clean_address"] = data["clean_address"].apply(lambda x: clean_address(x).replace(",", "").strip())

In [8]:
data["clean_address"] = data["clean_address"].apply(lambda x: x\
    .replace(" Г Г ", " Г ")\
    .replace("МОСКВА Г МОСКВА", "МОСКВА")
)

In [9]:
data.head(2)

,clean_address,address
0,МОСКВА УЛ СКАКОВАЯ 18,"РОССИЯ, МОСКВА Г, Г МОСКВА, УЛ СКАКОВАЯ, Д 18"
1,МОСКОВСКАЯ ОБЛ БЕГОВОЙ Р-Н Г МОСКВА УЛ СКАКОВА...,"РОССИЯ, МОСКОВСКАЯ ОБЛ, БЕГОВОЙ Р-Н, Г МОСКВА,..."


In [12]:
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [86]:
def get_building_type(address, driver):
    try:
        wait = WebDriverWait(driver, 10)
        # time.sleep(2.8)

        inp = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'input[placeholder="Поиск в 2ГИС"]')))
        inp.clear()
        inp.send_keys(address)
        wait_random()
        inp.send_keys(Keys.RETURN)
        wait_random()

        # можно заменить time.sleep на ожидание нужного контейнера
        # time.sleep(5)

        # контейнер с адресом/инфо
        info = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, 'div._ojs7nk, div._1x89xo5, div._1idnaau')))
        wait_random()

        # тип здания — первый span с классом _wrdavn или _oqoid (в вашем html это _wrdavn)
        spans = info.find_elements(By.CSS_SELECTOR, 'span._wrdavn, span._oqoid')
        print(spans)
        if not spans:
            raise NoSuchElementException('span with building type not found')
        building_type = spans[0].text.strip()

        # адрес — h1 > span (если не найдено, ищем div._1idnaau > span)
        try:
            h1_span = driver.find_element(By.CSS_SELECTOR, 'h1._1x89xo5 span')
            addr_text = h1_span.text.strip().replace('\u00A0', ' ')
        except NoSuchElementException:
            addr_span = driver.find_elements(By.CSS_SELECTOR, 'div._zjunba span._oqoid')
            addr_text = addr_span[0].text.strip().replace('\u00A0', ' ') if addr_span else "Не удалось определить адрес"

        # если несколько вариантов выбора
        

        # <div class="_1idnaau"><span class="_oqoid">Административное здание</span></div>

        print(building_type, addr_text)
        return [building_type, addr_text]

    except TimeoutException:
        return None
    except (NoSuchElementException, ElementNotInteractableException):
        return None

In [150]:
import requests
from urllib.parse import urlencode

def geocode_address(address: str, server_url: str = "http://localhost:8080"):
    """
    Отправляет запрос к Nominatim‑серверу и возвращает первый найденный результат.
    """
    # Параметры запроса согласно API Nominatim
    params = {
        "q": address,
        "format": "json",
        # "addressdetails": 1,
        "limit": 1,          # возвращаем только первый результат
        # "accept-language": "ru"  # ответы на русском, если поддерживается
    }

    url = f"{server_url}/search?{urlencode(params)}"

    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()
        return data[0] if data else None
    except requests.RequestException as e:
        # print(f"Ошибка запроса: {e}")
        return None

In [151]:
def osm_parser(address):
    response = geocode_address(address)
    if response != None:
        bulding_type = response["type"]
        bulding_class = response["class"]
        bulding_addresstype = response["addresstype"]
        return [bulding_type, bulding_class, bulding_addresstype]
    else:
        return None

In [161]:
data[87934:].head(20)

,clean_address,address,Тип_здания_2ГИС,Адрес_2ГИС,All_types
87934,ИРКУТСКАЯ ОБЛ Г ИРКУТСК УЛ ПОЛЯРНАЯ 117,"РОССИЯ, ИРКУТСКАЯ ОБЛ, Г ИРКУТСК, УЛ ПОЛЯРНАЯ,...",NaN,NaN,nan
87935,ПРИМОРСКИЙ КРАЙ Г ВЛАДИВОСТОК УЛ АДМИРАЛА ЮМАШ...,"РОССИЯ, ПРИМОРСКИЙ КРАЙ, Г ВЛАДИВОСТОК, УЛ АДМ...",NaN,NaN,nan
87936,ЛЕНИНГРАДСКАЯ ОБЛ ПРИОЗЕРСКИЙ Р-Н П МИЧУРИНСКО...,"РОССИЯ, ЛЕНИНГРАДСКАЯ ОБЛ, ПРИОЗЕРСКИЙ Р-Н, П ...",NaN,NaN,nan
87937,МОСКВА УЛ КИРОВОГРАДСКАЯ 13А 01,"РОССИЯ, МОСКВА Г, Г МОСКВА, УЛ КИРОВОГРАДСКАЯ,...",NaN,NaN,nan
87938,КРАСНОЯРСКИЙ КРАЙ Г КРАСНОЯРСК УЛ 9 МАЯ 59,"РОССИЯ, КРАСНОЯРСКИЙ КРАЙ, Г КРАСНОЯРСК, УЛ 9 ...",apartments,NaN,"['apartments', 'building', 'building']"
87939,РОСТОВСКАЯ ОБЛ АЗОВСКИЙ Р-Н С КУЛЕШОВКА УЛ ТАМ...,"РОССИЯ, РОСТОВСКАЯ ОБЛ, АЗОВСКИЙ Р-Н, С КУЛЕШО...",tertiary,NaN,"['tertiary', 'highway', 'road']"
87940,ОРЕНБУРГСКАЯ ОБЛ САРАКТАШСКИЙ Р-Н С ЧЕРКАССЫ У...,"РОССИЯ, ОРЕНБУРГСКАЯ ОБЛ, САРАКТАШСКИЙ Р-Н, С ...",NaN,NaN,nan
87941,РОСТОВСКАЯ ОБЛ Г РОСТОВ-НА-ДОНУ УЛ СОБИНО 130,"РОССИЯ, РОСТОВСКАЯ ОБЛ, Г РОСТОВ-НА-ДОНУ, УЛ С...",house,NaN,"['house', 'building', 'building']"
87942,ПРИМОРСКИЙ КРАЙ Г ВЛАДИВОСТОК УЛ СЕЛЬСКАЯ 8,"РОССИЯ, ПРИМОРСКИЙ КРАЙ, Г ВЛАДИВОСТОК, УЛ СЕЛ...",NaN,NaN,nan
87943,САХА (ЯКУТИЯ) РЕСП Г ЯКУТСК УЛ КУРНАТОВСКОГО 34,"РОССИЯ, САХА (ЯКУТИЯ) РЕСП, Г ЯКУТСК, УЛ КУРНА...",NaN,NaN,nan


ERROR! Session/line number was not unique in database. History logging moved to new session 106


In [195]:
for i in tqdm(range(87934, len(data)-1, 1)):
    if str(data.iloc[i]["Тип_здания_2ГИС"]) == "nan" or str(data.iloc[i]["Тип_здания_2ГИС"]) == 'None':
        address = str(data.iloc[i]["clean_address"])
        # print(index, address, sep="\t")
        
        address_data = osm_parser(address)
        
        if address_data is not None:
            data.at[i, "Тип_здания_2ГИС"] = address_data[0]
            data.at[i, "All_types"] = str(address_data)
        else:
            pass

  0%|          | 0/87934 [00:00<?, ?it/s]

In [192]:
data["Тип_здания_2ГИС"].astype("str").unique()

array(['nan', 'apartments', 'tertiary', 'house', 'residential',
       'detached', 'bus_stop', 'motel', 'hamlet', 'unclassified',
       'kindergarten', 'university', 'yes', 'newspaper', 'hotel',
       'supermarket', 'city', 'service', 'secondary', 'outpost',
       'dormitory', 'cosmetics', 'flooring', 'primary', 'waste_basket',
       'construction', 'commercial', 'allotments', 'administrative',
       'village', 'industrial', 'mall', 'quarter', 'hangar', 'company',
       'beauty', 'trunk', 'hospital', 'train_station', 'hairdresser',
       'office', 'arts_centre', 'car_parts', 'suburb', 'memorial', 'cafe',
       'living_street', 'retail', 'post_office', 'public', 'theatre',
       'school', 'convenience', 'guest_house', 'cinema', 'water_utility',
       'bakery', 'sports_centre', 'neighbourhood', 'ruins', 'works',
       'library', 'hostel', 'square', 'damaged', 'bar',
       'exhibition_centre', 'dance', 'track', 'courthouse',
       'driving_school', 'building', 'public_buildin

In [196]:
data.to_excel("промежуточный_4.xlsx")

In [197]:
data[data["Тип_здания_2ГИС"].notna()]

,clean_address,address,Тип_здания_2ГИС,Адрес_2ГИС,All_types
87938,КРАСНОЯРСКИЙ КРАЙ Г КРАСНОЯРСК УЛ 9 МАЯ 59,"РОССИЯ, КРАСНОЯРСКИЙ КРАЙ, Г КРАСНОЯРСК, УЛ 9 ...",apartments,NaN,"['apartments', 'building', 'building']"
87939,РОСТОВСКАЯ ОБЛ АЗОВСКИЙ Р-Н С КУЛЕШОВКА УЛ ТАМ...,"РОССИЯ, РОСТОВСКАЯ ОБЛ, АЗОВСКИЙ Р-Н, С КУЛЕШО...",tertiary,NaN,"['tertiary', 'highway', 'road']"
87941,РОСТОВСКАЯ ОБЛ Г РОСТОВ-НА-ДОНУ УЛ СОБИНО 130,"РОССИЯ, РОСТОВСКАЯ ОБЛ, Г РОСТОВ-НА-ДОНУ, УЛ С...",house,NaN,"['house', 'building', 'building']"
87962,ИРКУТСКАЯ ОБЛ Г ИРКУТСК УЛ БАРРИКАД 145/15,"РОССИЯ, ИРКУТСКАЯ ОБЛ, Г ИРКУТСК, УЛ БАРРИКАД,...",residential,NaN,"['residential', 'highway', 'road']"
87964,ПЕРМСКИЙ КРАЙ Г ПЕРМЬ УЛ КУЙБЫШЕВА 71/1,"РОССИЯ, ПЕРМСКИЙ КРАЙ, Г ПЕРМЬ, УЛ КУЙБЫШЕВА, ...",residential,NaN,"['residential', 'highway', 'road']"
...,...,...,...,...,...
175834,ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ Г НОВОЧЕБОКСАРС...,"РОССИЯ, ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ, Г НОВО...",apartments,None,"['apartments', 'building', 'building']"
175840,ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ Г НОВОЧЕБОКСАРС...,"РОССИЯ, ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ, Г НОВО...",secondary,None,"['secondary', 'highway', 'road']"
175842,РОСТОВСКАЯ ОБЛ Г РОСТОВ-НА-ДОНУ УЛ ЕВДОКИМОВА ...,"РОССИЯ, РОСТОВСКАЯ ОБЛ, Г РОСТОВ-НА-ДОНУ, УЛ Е...",alcohol,None,"['alcohol', 'shop', 'shop']"
175854,ТЮМЕНСКАЯ ОБЛ ТЮМЕНСКИЙ Р-Н ТУРАЕВА УЛ АШИРБЕК...,"РОССИЯ, ТЮМЕНСКАЯ ОБЛ, ТЮМЕНСКИЙ Р-Н, Д ТУРАЕВ...",residential,None,"['residential', 'highway', 'road']"


In [183]:
str(data.iloc[87934+12000]["Тип_здания_2ГИС"])

'nan'

In [198]:
data[87934:][data["Тип_здания_2ГИС"].notna()]

C:\Users\ararat\AppData\Local\Temp\ipykernel_8244\402563440.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data[87934:][data["Тип_здания_2ГИС"].notna()]


,clean_address,address,Тип_здания_2ГИС,Адрес_2ГИС,All_types
87938,КРАСНОЯРСКИЙ КРАЙ Г КРАСНОЯРСК УЛ 9 МАЯ 59,"РОССИЯ, КРАСНОЯРСКИЙ КРАЙ, Г КРАСНОЯРСК, УЛ 9 ...",apartments,NaN,"['apartments', 'building', 'building']"
87939,РОСТОВСКАЯ ОБЛ АЗОВСКИЙ Р-Н С КУЛЕШОВКА УЛ ТАМ...,"РОССИЯ, РОСТОВСКАЯ ОБЛ, АЗОВСКИЙ Р-Н, С КУЛЕШО...",tertiary,NaN,"['tertiary', 'highway', 'road']"
87941,РОСТОВСКАЯ ОБЛ Г РОСТОВ-НА-ДОНУ УЛ СОБИНО 130,"РОССИЯ, РОСТОВСКАЯ ОБЛ, Г РОСТОВ-НА-ДОНУ, УЛ С...",house,NaN,"['house', 'building', 'building']"
87962,ИРКУТСКАЯ ОБЛ Г ИРКУТСК УЛ БАРРИКАД 145/15,"РОССИЯ, ИРКУТСКАЯ ОБЛ, Г ИРКУТСК, УЛ БАРРИКАД,...",residential,NaN,"['residential', 'highway', 'road']"
87964,ПЕРМСКИЙ КРАЙ Г ПЕРМЬ УЛ КУЙБЫШЕВА 71/1,"РОССИЯ, ПЕРМСКИЙ КРАЙ, Г ПЕРМЬ, УЛ КУЙБЫШЕВА, ...",residential,NaN,"['residential', 'highway', 'road']"
...,...,...,...,...,...
175834,ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ Г НОВОЧЕБОКСАРС...,"РОССИЯ, ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ, Г НОВО...",apartments,None,"['apartments', 'building', 'building']"
175840,ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ Г НОВОЧЕБОКСАРС...,"РОССИЯ, ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ, Г НОВО...",secondary,None,"['secondary', 'highway', 'road']"
175842,РОСТОВСКАЯ ОБЛ Г РОСТОВ-НА-ДОНУ УЛ ЕВДОКИМОВА ...,"РОССИЯ, РОСТОВСКАЯ ОБЛ, Г РОСТОВ-НА-ДОНУ, УЛ Е...",alcohol,None,"['alcohol', 'shop', 'shop']"
175854,ТЮМЕНСКАЯ ОБЛ ТЮМЕНСКИЙ Р-Н ТУРАЕВА УЛ АШИРБЕК...,"РОССИЯ, ТЮМЕНСКАЯ ОБЛ, ТЮМЕНСКИЙ Р-Н, Д ТУРАЕВ...",residential,None,"['residential', 'highway', 'road']"


In [ ]:
data

In [10]:
87934-20000

67934

In [22]:
def get_building_type(address, driver):
    try:
        # wait = WebDriverWait(driver, 10)
        # time.sleep(2.8)

        inp = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'input[placeholder="Поиск в 2ГИС"]')))
        inp.clear()
        inp.send_keys(address)
        wait_random()
        inp.send_keys(Keys.RETURN)
        wait_random()

        # можно заменить time.sleep на ожидание нужного контейнера
        html = driver.page_source  
        return html
    except:
        return None

def get_oqoid_from_div(soup):
    results = []
    for div in soup.find_all("div", class_="_1idnaau"):
        span = div.find("span", class_="_oqoid")
        if span:
            results.append(span.get_text(strip=True))
    return results

In [31]:
def open_driver():
    driver = webdriver.Firefox()
    wait = WebDriverWait(driver, 10)
    driver.get('https://2gis.ru/')
    time.sleep(5)
    return driver
    
def close_driver(driver):
    driver.close()
    time.sleep(11)

In [19]:
data["Тип_здания_2ГИС"] = "nan"
data["All_types"] = "nan"

In [44]:
driver = webdriver.Firefox()
wait = WebDriverWait(driver, 10)
driver.get('https://2gis.ru/')
time.sleep(5)

In [45]:
for i in tqdm(range(67934+3510+5532, 87934-5000, 1)):
    if str(data.iloc[i]["Тип_здания_2ГИС"]) == "nan" or str(data.iloc[i]["Тип_здания_2ГИС"]) == 'None':
        address = str(data.iloc[i]["clean_address"])
        
        html = get_building_type(address, driver)

        if html is not None:
            soup = BeautifulSoup(html, "html.parser")
            types = get_oqoid_from_div(soup)
            if len(types) > 0:
                counter = Counter(types)
                most_common_item, freq = counter.most_common(1)[0]
                all_types = list(set(types))
                data.at[i, "Тип_здания_2ГИС"] = most_common_item
                data.at[i, "All_types"] = all_types
        else: 
            pass

  0%|          | 0/5958 [00:00<?, ?it/s]

In [46]:
data.to_excel("Арина_часть1.xlsx", index=False)

In [244]:
for i in tqdm(range(87934+10000+8293+6700+5953+4000+5452+4000+2416+3366, 132000+8000, 1)):
    if str(data.iloc[i]["Тип_здания_2ГИС"]) == "nan" or str(data.iloc[i]["Тип_здания_2ГИС"]) == 'None':
        address = str(data.iloc[i]["clean_address"])
        # print(index, address, sep="\t")

        html = get_building_type(address, driver)

        if html is not None:
            soup = BeautifulSoup(html, "html.parser")
            types = get_oqoid_from_div(soup)
            if len(types) > 0:
                counter = Counter(types)
                most_common_item, freq = counter.most_common(1)[0]
                all_types = list(set(types))
                data.at[i, "Тип_здания_2ГИС"] = most_common_item
                data.at[i, "All_types"] = all_types
        else: 
            pass
    
        # address_data = osm_parser(address)
        
        # if address_data is not None:
        #     data.at[i, "Тип_здания_2ГИС"] = address_data[0]
        #     data.at[i, "All_types"] = str(address_data)
        # else:
        #     pass

  0%|          | 0/1886 [00:00<?, ?it/s]

In [236]:
5452+1-1+3366

5452

In [228]:
len(data) - (87934/2)

131902.0

In [ ]:
5953

In [247]:
data.to_excel("промежуточный_11.xlsx", index=False)

In [ ]:
17688 

In [230]:
data[87934:][data["Тип_здания_2ГИС"].notna()]

C:\Users\ararat\AppData\Local\Temp\ipykernel_8244\402563440.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data[87934:][data["Тип_здания_2ГИС"].notna()]


,clean_address,address,Тип_здания_2ГИС,Адрес_2ГИС,All_types
87934,ИРКУТСКАЯ ОБЛ Г ИРКУТСК УЛ ПОЛЯРНАЯ 117,"РОССИЯ, ИРКУТСКАЯ ОБЛ, Г ИРКУТСК, УЛ ПОЛЯРНАЯ,...",Административное здание,NaN,"[Административное здание, Автосервис, Склад]"
87935,ПРИМОРСКИЙ КРАЙ Г ВЛАДИВОСТОК УЛ АДМИРАЛА ЮМАШ...,"РОССИЯ, ПРИМОРСКИЙ КРАЙ, Г ВЛАДИВОСТОК, УЛ АДМ...",Жилой дом,NaN,"[Административное здание, Автосервис, Склад]"
87936,ЛЕНИНГРАДСКАЯ ОБЛ ПРИОЗЕРСКИЙ Р-Н П МИЧУРИНСКО...,"РОССИЯ, ЛЕНИНГРАДСКАЯ ОБЛ, ПРИОЗЕРСКИЙ Р-Н, П ...",Частный дом,NaN,"[Административное здание, Автосервис, Склад]"
87938,КРАСНОЯРСКИЙ КРАЙ Г КРАСНОЯРСК УЛ 9 МАЯ 59,"РОССИЯ, КРАСНОЯРСКИЙ КРАЙ, Г КРАСНОЯРСК, УЛ 9 ...",apartments,NaN,"['apartments', 'building', 'building']"
87939,РОСТОВСКАЯ ОБЛ АЗОВСКИЙ Р-Н С КУЛЕШОВКА УЛ ТАМ...,"РОССИЯ, РОСТОВСКАЯ ОБЛ, АЗОВСКИЙ Р-Н, С КУЛЕШО...",tertiary,NaN,"['tertiary', 'highway', 'road']"
...,...,...,...,...,...
175834,ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ Г НОВОЧЕБОКСАРС...,"РОССИЯ, ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ, Г НОВО...",apartments,None,"['apartments', 'building', 'building']"
175840,ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ Г НОВОЧЕБОКСАРС...,"РОССИЯ, ЧУВАШСКАЯ РЕСПУБЛИКА - ЧУВАШИЯ, Г НОВО...",secondary,None,"['secondary', 'highway', 'road']"
175842,РОСТОВСКАЯ ОБЛ Г РОСТОВ-НА-ДОНУ УЛ ЕВДОКИМОВА ...,"РОССИЯ, РОСТОВСКАЯ ОБЛ, Г РОСТОВ-НА-ДОНУ, УЛ Е...",alcohol,None,"['alcohol', 'shop', 'shop']"
175854,ТЮМЕНСКАЯ ОБЛ ТЮМЕНСКИЙ Р-Н ТУРАЕВА УЛ АШИРБЕК...,"РОССИЯ, ТЮМЕНСКАЯ ОБЛ, ТЮМЕНСКИЙ Р-Н, Д ТУРАЕВ...",residential,None,"['residential', 'highway', 'road']"


In [ ]:
87934+10000

In [216]:
data[87934:87934+10000][data["Тип_здания_2ГИС"].notna()]

C:\Users\ararat\AppData\Local\Temp\ipykernel_8244\3834734753.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  data[87934:87934+10000][data["Тип_здания_2ГИС"].notna()]


,clean_address,address,Тип_здания_2ГИС,Адрес_2ГИС,All_types
87934,ИРКУТСКАЯ ОБЛ Г ИРКУТСК УЛ ПОЛЯРНАЯ 117,"РОССИЯ, ИРКУТСКАЯ ОБЛ, Г ИРКУТСК, УЛ ПОЛЯРНАЯ,...",Административное здание,NaN,"[Административное здание, Автосервис, Склад]"
87935,ПРИМОРСКИЙ КРАЙ Г ВЛАДИВОСТОК УЛ АДМИРАЛА ЮМАШ...,"РОССИЯ, ПРИМОРСКИЙ КРАЙ, Г ВЛАДИВОСТОК, УЛ АДМ...",Жилой дом,NaN,"[Административное здание, Автосервис, Склад]"
87936,ЛЕНИНГРАДСКАЯ ОБЛ ПРИОЗЕРСКИЙ Р-Н П МИЧУРИНСКО...,"РОССИЯ, ЛЕНИНГРАДСКАЯ ОБЛ, ПРИОЗЕРСКИЙ Р-Н, П ...",Частный дом,NaN,"[Административное здание, Автосервис, Склад]"
87938,КРАСНОЯРСКИЙ КРАЙ Г КРАСНОЯРСК УЛ 9 МАЯ 59,"РОССИЯ, КРАСНОЯРСКИЙ КРАЙ, Г КРАСНОЯРСК, УЛ 9 ...",apartments,NaN,"['apartments', 'building', 'building']"
87939,РОСТОВСКАЯ ОБЛ АЗОВСКИЙ Р-Н С КУЛЕШОВКА УЛ ТАМ...,"РОССИЯ, РОСТОВСКАЯ ОБЛ, АЗОВСКИЙ Р-Н, С КУЛЕШО...",tertiary,NaN,"['tertiary', 'highway', 'road']"
...,...,...,...,...,...
97892,НИЖЕГОРОДСКАЯ ОБЛ Г НИЖНИЙ НОВГОРОД УЛ ОТЕЧЕСТ...,"РОССИЯ, НИЖЕГОРОДСКАЯ ОБЛ, Г НИЖНИЙ НОВГОРОД, ...",house,NaN,"['house', 'building', 'building']"
97920,МОСКВА УЛ ГЕНЕРАЛА БЕЛОБОРОДОВА 4,"РОССИЯ, МОСКВА Г, Г МОСКВА, УЛ ГЕНЕРАЛА БЕЛОБО...",kindergarten,NaN,"['kindergarten', 'building', 'building']"
97924,МОСКВА УЛ ИНЖЕНЕРНАЯ 13,"РОССИЯ, МОСКВА Г, Г МОСКВА, УЛ ИНЖЕНЕРНАЯ, Д 13",apartments,NaN,"['apartments', 'building', 'building']"
97928,ЛЕНИНГРАДСКАЯ ОБЛ ЛОМОНОСОВСКИЙ Р-Н ЛОПУХИНКА ...,"РОССИЯ, ЛЕНИНГРАДСКАЯ ОБЛ, ЛОМОНОСОВСКИЙ Р-Н, ...",apartments,NaN,"['apartments', 'building', 'building']"


In [217]:
8985*100/10000 = 89.85%

89.85

In [282]:
data

,clean_address,address,Тип_здания_2ГИС,Адрес_2ГИС,All_types
0,МОСКВА УЛ СКАКОВАЯ 18,"РОССИЯ, МОСКВА Г, Г МОСКВА, УЛ СКАКОВАЯ, Д 18",NaN,NaN,nan
1,МОСКОВСКАЯ ОБЛ БЕГОВОЙ Р-Н Г МОСКВА УЛ СКАКОВА...,"РОССИЯ, МОСКОВСКАЯ ОБЛ, БЕГОВОЙ Р-Н, Г МОСКВА,...",NaN,NaN,nan
2,МОСКОВСКАЯ ОБЛ Г МОСКВА УЛ СКАКОВАЯ 18,"РОССИЯ, МОСКОВСКАЯ ОБЛ, Г МОСКВА, УЛ СКАКОВАЯ,...",NaN,NaN,nan
3,МОСКОВСКАЯ ОБЛ БЕРЕГОВОЙ Р-Н Г МОСКВА УЛ СКАКО...,"РОССИЯ, МОСКОВСКАЯ ОБЛ, БЕРЕГОВОЙ Р-Н, Г МОСКВ...",NaN,NaN,nan
4,МОСКОВСКАЯ ОБЛ Г МОСКВА УЛ СКАКОВАЯ Д. 18,"РОССИЯ, МОСКОВСКАЯ ОБЛ, Г МОСКВА, УЛ СКАКОВАЯ,...",NaN,NaN,nan
...,...,...,...,...,...
175864,ЧЕЛЯБИНСКАЯ ОБЛ Г МАГНИТОГОРСК УЛ ЭЛЕВАТОРНАЯ 13,"РОССИЯ, ЧЕЛЯБИНСКАЯ ОБЛ, Г МАГНИТОГОРСК, УЛ ЭЛ...",None,None,nan
175865,КУРСКАЯ ОБЛ Г ЛЬГОВ УЛ К.МАРКСА 116,"РОССИЯ, КУРСКАЯ ОБЛ, Г ЛЬГОВ, УЛ К.МАРКСА, Д 116",None,None,nan
175866,ПЕНЗЕНСКАЯ ОБЛ Г ПЕНЗА УЛ КАРПИНСКОГО 27,"РОССИЯ, ПЕНЗЕНСКАЯ ОБЛ, Г ПЕНЗА, УЛ КАРПИНСКОГ...",apartments,None,"['apartments', 'building', 'building']"
175867,ХАНТЫ-МАНСИЙСКИЙ АВТОНОМНЫЙ ОКРУГ - ЮГРА АО Г ...,"РОССИЯ, ХАНТЫ-МАНСИЙСКИЙ АВТОНОМНЫЙ ОКРУГ - ЮГ...",None,None,nan


In [246]:
data.to_excel()

,clean_address,address,Тип_здания_2ГИС,Адрес_2ГИС,All_types
0,МОСКВА УЛ СКАКОВАЯ 18,"РОССИЯ, МОСКВА Г, Г МОСКВА, УЛ СКАКОВАЯ, Д 18",NaN,NaN,nan
1,МОСКОВСКАЯ ОБЛ БЕГОВОЙ Р-Н Г МОСКВА УЛ СКАКОВА...,"РОССИЯ, МОСКОВСКАЯ ОБЛ, БЕГОВОЙ Р-Н, Г МОСКВА,...",NaN,NaN,nan
2,МОСКОВСКАЯ ОБЛ Г МОСКВА УЛ СКАКОВАЯ 18,"РОССИЯ, МОСКОВСКАЯ ОБЛ, Г МОСКВА, УЛ СКАКОВАЯ,...",NaN,NaN,nan
3,МОСКОВСКАЯ ОБЛ БЕРЕГОВОЙ Р-Н Г МОСКВА УЛ СКАКО...,"РОССИЯ, МОСКОВСКАЯ ОБЛ, БЕРЕГОВОЙ Р-Н, Г МОСКВ...",NaN,NaN,nan
4,МОСКОВСКАЯ ОБЛ Г МОСКВА УЛ СКАКОВАЯ Д. 18,"РОССИЯ, МОСКОВСКАЯ ОБЛ, Г МОСКВА, УЛ СКАКОВАЯ,...",NaN,NaN,nan
...,...,...,...,...,...
175864,ЧЕЛЯБИНСКАЯ ОБЛ Г МАГНИТОГОРСК УЛ ЭЛЕВАТОРНАЯ 13,"РОССИЯ, ЧЕЛЯБИНСКАЯ ОБЛ, Г МАГНИТОГОРСК, УЛ ЭЛ...",None,None,nan
175865,КУРСКАЯ ОБЛ Г ЛЬГОВ УЛ К.МАРКСА 116,"РОССИЯ, КУРСКАЯ ОБЛ, Г ЛЬГОВ, УЛ К.МАРКСА, Д 116",None,None,nan
175866,ПЕНЗЕНСКАЯ ОБЛ Г ПЕНЗА УЛ КАРПИНСКОГО 27,"РОССИЯ, ПЕНЗЕНСКАЯ ОБЛ, Г ПЕНЗА, УЛ КАРПИНСКОГ...",apartments,None,"['apartments', 'building', 'building']"
175867,ХАНТЫ-МАНСИЙСКИЙ АВТОНОМНЫЙ ОКРУГ - ЮГРА АО Г ...,"РОССИЯ, ХАНТЫ-МАНСИЙСКИЙ АВТОНОМНЫЙ ОКРУГ - ЮГ...",None,None,nan


In [283]:
data2 = pd.read_excel("промежуточный_6_6.xlsx", dtype=str)

In [284]:
data2.drop(columns = {"Unnamed: 0.2", "Unnamed: 0.1", "Unnamed: 0"},inplace=True)

In [269]:
data2["All_types"].unique()

array([nan, "['Административное здание', 'Автосервис', 'Склад']",
       "['apartments', 'building', 'building']", ...,
       "['research', 'office', 'office']",
       "['terminal', 'aeroway', 'aeroway']",
       "['sports_hall', 'leisure', 'leisure']"],
      shape=(1661,), dtype=object)

In [270]:
data2_df = data2[data2["Тип_здания_2ГИС"].notna()]

In [ ]:
data2_df["All_types"].apply(lambda x: len(x))

In [ ]:
data2_df

In [ ]:
data2_df['types_parsed'] = (
    data2_df['All_types']
      .str.strip("[]")          # убираем внешние квадратные скобки
      .str.replace("'", "", regex=False)  # убираем кавычки
      .str.split(", ")          # делим по запятой + пробелу
)

# Объединение

In [288]:
data["All_types"] = data["All_types"].astype("str")

In [289]:
data2["All_types"] = data2["All_types"].astype("str")

In [290]:
data.head(2)

,clean_address,address,Тип_здания_2ГИС,Адрес_2ГИС,All_types
0,МОСКВА УЛ СКАКОВАЯ 18,"РОССИЯ, МОСКВА Г, Г МОСКВА, УЛ СКАКОВАЯ, Д 18",NaN,NaN,nan
1,МОСКОВСКАЯ ОБЛ БЕГОВОЙ Р-Н Г МОСКВА УЛ СКАКОВА...,"РОССИЯ, МОСКОВСКАЯ ОБЛ, БЕГОВОЙ Р-Н, Г МОСКВА,...",NaN,NaN,nan


In [291]:
data2.head(2)

,clean_address,address,Тип_здания_2ГИС,Адрес_2ГИС,All_types
0,МОСКВА УЛ СКАКОВАЯ 18,"РОССИЯ, МОСКВА Г, Г МОСКВА, УЛ СКАКОВАЯ, Д 18",NaN,NaN,nan
1,МОСКОВСКАЯ ОБЛ БЕГОВОЙ Р-Н Г МОСКВА УЛ СКАКОВА...,"РОССИЯ, МОСКОВСКАЯ ОБЛ, БЕГОВОЙ Р-Н, Г МОСКВА,...",NaN,NaN,nan


In [296]:
len(data[data["Тип_здания_2ГИС"].notna()])

54648

In [295]:
len(data2[data2["Тип_здания_2ГИС"].notna()])

43705

In [298]:
len(data)/2

87934.5

In [372]:
mer = pd.merge(data, data2, how="inner", on="clean_address")

In [386]:
mer["index"] = mer.index

In [398]:
mer = mer.drop(columns={"address_y"}).rename(columns={"address_x":"address"})

In [399]:
mer = mer[[
    'index',
    'clean_address', 'address', 'Тип_здания_2ГИС_x', 'Адрес_2ГИС_x',
       'All_types_x', 'Тип_здания_2ГИС_y', 'Адрес_2ГИС_y',
       'All_types_y'
]]

In [401]:
mer["Тип_здания_2ГИС_x"] = mer["Тип_здания_2ГИС_x"].astype("str")
mer["Тип_здания_2ГИС_y"] = mer["Тип_здания_2ГИС_y"].astype("str")

mer["Адрес_2ГИС_x"] = mer["Адрес_2ГИС_x"].astype("str")
mer["Адрес_2ГИС_y"] = mer["Адрес_2ГИС_y"].astype("str")

mer["All_types_x"] = mer["All_types_x"].astype("str")
mer["All_types_y"] = mer["All_types_y"].astype("str")

In [402]:
mer['Тип_здания_2ГИС_x'] = mer['Тип_здания_2ГИС_x'].replace('nan', None)
mer['Тип_здания_2ГИС_y'] = mer['Тип_здания_2ГИС_y'].replace('nan', None)

mer['Адрес_2ГИС_x'] = mer['Адрес_2ГИС_x'].replace('nan', None)
mer['Адрес_2ГИС_y'] = mer['Адрес_2ГИС_y'].replace('nan', None)

mer['All_types_x'] = mer['All_types_x'].replace('nan', None)
mer['All_types_y'] = mer['All_types_y'].replace('nan', None)

In [427]:
mer[175898:175899+1]

,index,clean_address,address,Тип_здания_2ГИС_x,Адрес_2ГИС_x,All_types_x,Тип_здания_2ГИС_y,Адрес_2ГИС_y,All_types_y,тип_здания,адрес_парсинга,all_types
175898,175898,ЧЕЛЯБИНСКАЯ ОБЛ СОСНОВСКИЙ Р-Н П НОВЫЙ КРЕМЕНК...,"РОССИЯ, ЧЕЛЯБИНСКАЯ ОБЛ, СОСНОВСКИЙ Р-Н, П НОВ...",None,None,None,Таунхаус,None,['Таунхаус'],Таунхаус,None,['Таунхаус']
175899,175899,ЧЕЛЯБИНСКАЯ ОБЛ Г МАГНИТОГОРСК УЛ ЭЛЕВАТОРНАЯ 13,"РОССИЯ, ЧЕЛЯБИНСКАЯ ОБЛ, Г МАГНИТОГОРСК, УЛ ЭЛ...",None,None,None,Частный дом,None,['Частный дом'],Частный дом,None,['Частный дом']


In [422]:
mer["тип_здания"] = None
mer["адрес_парсинга"] = None
mer["all_types"] = None

In [426]:
none_list = ["None", "nan", "NaN", "NoNe", "NAN"]

for index, row in tqdm(mer.iterrows()):
    
    type1 = str(row["Тип_здания_2ГИС_x"])
    type2 = str(row["Тип_здания_2ГИС_y"])
    building_type = type1 if type1 not in none_list else type2


    addr1 = str(row["Адрес_2ГИС_x"])
    addr2 = str(row["Адрес_2ГИС_y"])
    address = addr1 if addr1 not in none_list else addr2

    all_types1 = str(row["All_types_x"])
    all_types2 = str(row["All_types_y"])
    all_types = all_types1 if all_types1 not in none_list else all_types2

    mer.at[index, "тип_здания"] = building_type
    mer.at[index, "адрес_парсинга"] = address
    mer.at[index, "all_types"] = all_types

0it [00:00, ?it/s]

In [605]:
res = mer[87934:]

In [606]:
res = res[['index', 'clean_address', 'address', 'тип_здания', 'адрес_парсинга', 'all_types']]

In [607]:
res['lst_all_types'] = (
    res['all_types']
      .str.strip("[]")          # убираем внешние квадратные скобки
      .str.replace("'", "", regex=False)  # убираем кавычки
      .str.split(", ")          # делим по запятой + пробелу
)

In [608]:
res

,index,clean_address,address,тип_здания,адрес_парсинга,all_types,lst_all_types
87934,87934,ПРИМОРСКИЙ КРАЙ Г ВЛАДИВОСТОК УЛ 2-Я ПОСЕЛКОВА...,"РОССИЯ, ПРИМОРСКИЙ КРАЙ, Г ВЛАДИВОСТОК, УЛ 2-Я...",None,None,None,[None]
87935,87935,МОСКВА ПРОЕЗД МУКОМОЛЬНЫЙ 1 КОРП 1,"РОССИЯ, МОСКВА Г, Г МОСКВА, ПРОЕЗД МУКОМОЛЬНЫЙ...",None,None,None,[None]
87936,87936,ТЮМЕНСКАЯ ОБЛ Г ТОБОЛЬСК ВОСТОЧНЫЙ ПРОМЫШЛЕННЫ...,"РОССИЯ, ТЮМЕНСКАЯ ОБЛ, Г ТОБОЛЬСК, ВОСТОЧНЫЙ П...",None,None,None,[None]
87937,87937,САХАЛИНСКАЯ ОБЛ Г ЮЖНО-САХАЛИНСК УЛ СОВЕТСКАЯ 8Д,"РОССИЯ, САХАЛИНСКАЯ ОБЛ, Г ЮЖНО-САХАЛИНСК, УЛ ...",None,None,None,[None]
87938,87938,МОСКВА УЛ ФЕРГАНСКАЯ 24,"РОССИЯ, МОСКВА Г, Г МОСКВА, УЛ ФЕРГАНСКАЯ, Д 24",None,None,None,[None]
...,...,...,...,...,...,...,...
175899,175899,ЧЕЛЯБИНСКАЯ ОБЛ Г МАГНИТОГОРСК УЛ ЭЛЕВАТОРНАЯ 13,"РОССИЯ, ЧЕЛЯБИНСКАЯ ОБЛ, Г МАГНИТОГОРСК, УЛ ЭЛ...",Частный дом,None,['Частный дом'],[Частный дом]
175900,175900,КУРСКАЯ ОБЛ Г ЛЬГОВ УЛ К.МАРКСА 116,"РОССИЯ, КУРСКАЯ ОБЛ, Г ЛЬГОВ, УЛ К.МАРКСА, Д 116",Частный дом,None,['Частный дом'],[Частный дом]
175901,175901,ПЕНЗЕНСКАЯ ОБЛ Г ПЕНЗА УЛ КАРПИНСКОГО 27,"РОССИЯ, ПЕНЗЕНСКАЯ ОБЛ, Г ПЕНЗА, УЛ КАРПИНСКОГ...",apartments,None,"['apartments', 'building', 'building']","[apartments, building, building]"
175902,175902,ХАНТЫ-МАНСИЙСКИЙ АВТОНОМНЫЙ ОКРУГ - ЮГРА АО Г ...,"РОССИЯ, ХАНТЫ-МАНСИЙСКИЙ АВТОНОМНЫЙ ОКРУГ - ЮГ...",Жилой дом,None,['Жилой дом'],[Жилой дом]


In [589]:
list_types = res["lst_all_types"].tolist()

In [ ]:
list_types

In [591]:
flattened_list = [item for sublist in list_types for item in sublist]

In [592]:
flattened_list = list(set(flattened_list))

In [593]:
def drop_ru_elements(lst, pattern=r'[а-я]', exclude = False):
    """
    Исключает из списка все элементы, которые являются строками на русском языке.
    Аргумент:
        список (list) - Список для фильтрации.
        шаблон (str) - шаблон поиска (по умолчанию - любой символ).
    Возвращает:
        Список с исключенными элементами.
    """
    
    filtered_lst = []
    for item in lst:
        if not isinstance(item, str) or re.search(pattern, str(item).lower()) is None: # Если не строка или если нет русских букв
            # добавляем в список только те элементы, что соответствуют условиям
            filtered_lst.append(item)
    return filtered_lst

In [475]:
uniq_osm_types = drop_ru_elements(flattened_list)

In [487]:
df_uniq_osm_types = pd.DataFrame({"building_type":uniq_osm_types})

In [495]:
types_natcher = pd.read_excel("new_types.xlsx", sheet_name="типы_зданий", dtype="str")

In [496]:
types_natcher

,тип,ключ,значение_en,значение_ru,комментарий
0,Жилые,building,apartments,Многоквартирный жилой дом,Многоквартирный жилой дом
1,Жилые,building,barracks,Казармы,Казармы
2,Жилые,building,bungalow,Небольшой дом дачный,"Небольшой одноэтажный летний домик, дача"
3,Жилые,building,cabin,Небольшой дом по типу бытовки,cabin
4,Жилые,building,detached,Жилой дом,"Частный отдельный жилой дом, обычно на одну семью"
...,...,...,...,...,...
98,Прочие,building,tent,Палатка,"Стационарная палатка, тент."
99,Прочие,building,tower,Башня,Здание-башня.
100,Прочие,building,triumphal_arch,Триумфальная арка,Триумфальная арка (отдельно стоящее монументал...
101,Прочие,building,windmill,Мельница,"Здание, построенное как традиционная ветряная ..."


In [609]:
res = pd.merge(res, types_natcher[["тип", "значение_en", "значение_ru"]], how="left", left_on="тип_здания", right_on="значение_en").drop(columns={"значение_en"})

In [610]:
for index, row in tqdm(res.iterrows()):
    if str(row["значение_ru"]) not in none_list:
        type_ru = row["значение_ru"]
        
        res.at[index, "тип_здания"] = type_ru

0it [00:00, ?it/s]

In [611]:
res.head(20)

,index,clean_address,address,тип_здания,адрес_парсинга,all_types,lst_all_types,тип,значение_ru
0,87934,ПРИМОРСКИЙ КРАЙ Г ВЛАДИВОСТОК УЛ 2-Я ПОСЕЛКОВА...,"РОССИЯ, ПРИМОРСКИЙ КРАЙ, Г ВЛАДИВОСТОК, УЛ 2-Я...",None,None,None,[None],NaN,NaN
1,87935,МОСКВА ПРОЕЗД МУКОМОЛЬНЫЙ 1 КОРП 1,"РОССИЯ, МОСКВА Г, Г МОСКВА, ПРОЕЗД МУКОМОЛЬНЫЙ...",None,None,None,[None],NaN,NaN
2,87936,ТЮМЕНСКАЯ ОБЛ Г ТОБОЛЬСК ВОСТОЧНЫЙ ПРОМЫШЛЕННЫ...,"РОССИЯ, ТЮМЕНСКАЯ ОБЛ, Г ТОБОЛЬСК, ВОСТОЧНЫЙ П...",None,None,None,[None],NaN,NaN
3,87937,САХАЛИНСКАЯ ОБЛ Г ЮЖНО-САХАЛИНСК УЛ СОВЕТСКАЯ 8Д,"РОССИЯ, САХАЛИНСКАЯ ОБЛ, Г ЮЖНО-САХАЛИНСК, УЛ ...",None,None,None,[None],NaN,NaN
4,87938,МОСКВА УЛ ФЕРГАНСКАЯ 24,"РОССИЯ, МОСКВА Г, Г МОСКВА, УЛ ФЕРГАНСКАЯ, Д 24",None,None,None,[None],NaN,NaN
5,87939,ПРИМОРСКИЙ КРАЙ МИХАЙЛОВСКИЙ Р-Н С ПЕРВОМАЙСКО...,"РОССИЯ, ПРИМОРСКИЙ КРАЙ, МИХАЙЛОВСКИЙ Р-Н, С П...",None,None,None,[None],NaN,NaN
6,87940,ЛЕНИНГРАДСКАЯ ОБЛ ТОСНЕНСКИЙ Р-Н Г ТОСНО УЛ 2-...,"РОССИЯ, ЛЕНИНГРАДСКАЯ ОБЛ, ТОСНЕНСКИЙ Р-Н, Г Т...",None,None,None,[None],NaN,NaN
7,87941,ТАМБОВСКАЯ ОБЛ Г РАССКАЗОВО УЛ ПОСЕЛОК МЕХОВОЙ...,"РОССИЯ, ТАМБОВСКАЯ ОБЛ, Г РАССКАЗОВО, УЛ ПОСЕЛ...",None,None,None,[None],NaN,NaN
8,87942,ЛИПЕЦКАЯ ОБЛ Г ЛИПЕЦК ПР-КТ ПОБЕДЫ 74,"РОССИЯ, ЛИПЕЦКАЯ ОБЛ, Г ЛИПЕЦК, ПР-КТ ПОБЕДЫ, ...",None,None,None,[None],NaN,NaN
9,87943,ВЛАДИМИРСКАЯ ОБЛ Г КОВРОВ ПРОЕЗД ВОСТОЧНЫЙ 14/2,"РОССИЯ, ВЛАДИМИРСКАЯ ОБЛ, Г КОВРОВ, ПРОЕЗД ВОС...",None,None,None,[None],NaN,NaN


In [612]:
from pydantic import BaseModel 

class DateFormatter(BaseModel):
    original: str
    type: str


from openai import OpenAI
import json

# Initialize OpenAI client that points to the local LM Studio server
client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio"
)

In [613]:
def get_completion(text):
    messages = [
        {
            "role": "system", 
            "content": "Строго следуй инструкциям. Отвечай на русском языке."
        },
        {
            "role": "user", 
            "content": [
                {
                    "type": "text",
                    "text": f"""
    
                    Твоя задача соотнести этот тип здания >>> {text} <<< к одной из следующих категорий:
                        - Жилые
                        - Административные
                        - Муниципальные
                        - Коммерческие
                        - Религиозные
                        - Общественные
                        - Спортивные
                        - Хранение
                        - Энергетические и технические здания
                        - Прочие
                    """
                }
            ]
        }
    ]
    
    response = client.chat.completions.parse(
        model="google/gemma-3-4b",
        messages=messages,
        response_format=DateFormatter,
    )
    return json.loads(response.choices[0].message.content)   

In [614]:
json.loads(response.choices[0].message.content)

{'original': 'Частный дом', 'type': 'Жилые'}

In [615]:
get_completion("Частный дом	")["type"]

'Жилые'

In [631]:
uniq_types_ru = list(set(res["тип_здания"][
    (res["тип_здания"] != "None") &
    (res["тип"].isna())
].tolist()))

In [635]:
df_ru_types = pd.DataFrame({"тип_здания":uniq_types_ru})

In [637]:
for index, row in tqdm(df_ru_types.iterrows()):
    type_ru = row["тип_здания"]
    class_ru = get_completion(type_ru)["type"]
        
    df_ru_types.at[index, "тип"] = class_ru

0it [00:00, ?it/s]

In [648]:
df_ru_types.to_excel("2gis_classes.xlsx", index=False)

In [649]:
for index, row in tqdm(res.iterrows()):
    if str(row["тип_здания"]) not in none_list and str(row["тип"]) in none_list:
        
        type_ru = row["тип_здания"]
        class_ru = df_ru_types["тип"][df_ru_types["тип_здания"] == type_ru].iloc[0]

        res.at[index, "тип"] = class_ru

0it [00:00, ?it/s]

In [652]:
res[res["тип_здания"] == "None"]

,index,clean_address,address,тип_здания,адрес_парсинга,all_types,lst_all_types,тип,значение_ru
0,87934,ПРИМОРСКИЙ КРАЙ Г ВЛАДИВОСТОК УЛ 2-Я ПОСЕЛКОВА...,"РОССИЯ, ПРИМОРСКИЙ КРАЙ, Г ВЛАДИВОСТОК, УЛ 2-Я...",None,None,None,[None],NaN,NaN
1,87935,МОСКВА ПРОЕЗД МУКОМОЛЬНЫЙ 1 КОРП 1,"РОССИЯ, МОСКВА Г, Г МОСКВА, ПРОЕЗД МУКОМОЛЬНЫЙ...",None,None,None,[None],NaN,NaN
2,87936,ТЮМЕНСКАЯ ОБЛ Г ТОБОЛЬСК ВОСТОЧНЫЙ ПРОМЫШЛЕННЫ...,"РОССИЯ, ТЮМЕНСКАЯ ОБЛ, Г ТОБОЛЬСК, ВОСТОЧНЫЙ П...",None,None,None,[None],NaN,NaN
3,87937,САХАЛИНСКАЯ ОБЛ Г ЮЖНО-САХАЛИНСК УЛ СОВЕТСКАЯ 8Д,"РОССИЯ, САХАЛИНСКАЯ ОБЛ, Г ЮЖНО-САХАЛИНСК, УЛ ...",None,None,None,[None],NaN,NaN
4,87938,МОСКВА УЛ ФЕРГАНСКАЯ 24,"РОССИЯ, МОСКВА Г, Г МОСКВА, УЛ ФЕРГАНСКАЯ, Д 24",None,None,None,[None],NaN,NaN
...,...,...,...,...,...,...,...,...,...
87947,175881,ЧЕЛЯБИНСКАЯ ОБЛ Г КОПЕЙСК ПР-КТ СЛАВЫ 25,"РОССИЯ, ЧЕЛЯБИНСКАЯ ОБЛ, Г КОПЕЙСК, ПР-КТ СЛАВ...",None,None,None,[None],NaN,NaN
87948,175882,ТАТАРСТАН РЕСП НИЖНЕКАМСКИЙ Р-Н Г НИЖНЕКАМСК У...,"РОССИЯ, ТАТАРСТАН РЕСП, НИЖНЕКАМСКИЙ Р-Н, Г НИ...",None,None,None,[None],NaN,NaN
87949,175883,ДОНЕЦКАЯ НАРОДНАЯ РЕСП Г ТОРЕЗ УЛ ДНЕПРОВСКАЯ 21,"РОССИЯ, ДОНЕЦКАЯ НАРОДНАЯ РЕСП, Г ТОРЕЗ, УЛ ДН...",None,None,None,[None],NaN,NaN
87950,175884,КАРЕЛИЯ РЕСП КОНДОПОЖСКИЙ Р-Н Г КОНДОПОГА УЛ П...,"РОССИЯ, КАРЕЛИЯ РЕСП, КОНДОПОЖСКИЙ Р-Н, Г КОНД...",None,None,None,[None],NaN,NaN


In [669]:
driver = webdriver.Firefox()
wait = WebDriverWait(driver, 10)
driver.get('https://2gis.ru/')
time.sleep(5)

In [661]:
len(res[res["тип_здания"] == "None"])

8948

In [670]:
for index, row in tqdm(res[res["тип_здания"] == "None"].iterrows()):
    address = str(row["clean_address"])
    
    html = get_building_type(address, driver)

    if html is not None:
        soup = BeautifulSoup(html, "html.parser")
        types = get_oqoid_from_div(soup)
        if len(types) > 0:
            counter = Counter(types)
            most_common_item, freq = counter.most_common(1)[0]
            all_types = list(set(types))
            res.at[index, "тип_здания"] = most_common_item
            res.at[index, "all_types"] = all_types
    else: 
        pass

0it [00:00, ?it/s]

In [671]:
len(res[res["тип_здания"] == "None"])

2996

In [666]:
8947-490

8457

In [673]:
res.to_excel("Результат_парсинга_2ГИС.xlsx", index=False)